# TrustOCT Orchestration Notebook
### Title: TrustOCT: A Trustworthy Cross-Device Retinal OCT Disease Classification Framework Using Domain Generalization and Uncertainty-Aware Selective Prediction

This master notebook acts as the orchestrator for Google Colab GPU training, external validation, visual attributions, and calibration metrics calculations.

In [ ]:
# Cell 1 — Project Information
print("="*60)
print("TrustOCT")
print("Trustworthy Cross-Device Retinal OCT Classification")
print("="*60)

In [ ]:
# Cell 2 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 3 — Clone Repository
# Replace with your Git repository url
!git clone https://github.com/Gnanapravallika/Trustworthy-OCT-AI.git
%cd Trustworthy-OCT-AI
!git pull origin main

In [ ]:
# Cell 4 — Install Dependencies
!pip install -r requirements.txt

In [ ]:
# Cell 5 — Kaggle Authentication
from google.colab import files
files.upload()
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
# Cell 6 — Download Datasets
# 1. Kermany (Download from Kaggle)
!kaggle datasets download -d paultimothymooney/kermany2018 --unzip -p datasets/Kermany

# 2. OCTID (Copy from Mounted Google Drive)
# Adjust the path below to match where your OCTID folder is located on Google Drive
!mkdir -p datasets/OCTID
!cp -r "/content/drive/MyDrive/OCTID/." datasets/OCTID/

In [ ]:
# Cell 7 — Verify Dataset
!python -c "from src.datasets import verify_dataset; verify_dataset('configs/dataset.yaml', 'outputs/reports/dataset_report.json')"

In [ ]:
# Cell 8 — Dataset Statistics
from src.datasets import generate_statistics_report
generate_statistics_report('configs/dataset.yaml', 'outputs/reports/dataset_statistics.json', max_samples=1000)

In [ ]:
# Cell 9 — Baseline Training
!python train.py \
    --experiment_name EXP001_Baseline \
    --model_config configs/EXP001_Baseline.yaml

In [ ]:
# Cell 10 — MultiScale
!python train.py \
    --experiment_name EXP002_MultiScale \
    --model_config configs/EXP002_MultiScale.yaml

In [ ]:
# Cell 11 — CBAM
!python train.py \
    --experiment_name EXP003_CBAM \
    --model_config configs/EXP003_CBAM.yaml

In [ ]:
# Cell 12 — MixStyle
!python train.py \
    --experiment_name EXP004_MixStyle \
    --model_config configs/EXP004_MixStyle.yaml

In [ ]:
# Cell 13 — EDL
!python train.py \
    --experiment_name EXP005_EDL \
    --model_config configs/EXP005_EDL.yaml

In [ ]:
# Cell 14 — TrustOCT
!python train.py \
    --experiment_name EXP006_TrustOCT \
    --model_config configs/EXP006_TrustOCT.yaml

In [ ]:
# Cell 15 — External Validation
import torch
from src.models import build_model
from src.datasets import get_dataset_and_loader
from src.evaluation import evaluate_cross_dataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load best full model weights
model = build_model('configs/EXP006_TrustOCT.yaml')
checkpoint = torch.load('experiments/EXP006_TrustOCT/weights_best.pth', map_location=device)
model.load_state_dict(checkpoint['model_state'])
model = model.to(device)

# Setup external dataset loader (e.g. Test split of Kermany or mapped target loaders)
_, test_loader = get_dataset_and_loader('val', 'configs/dataset.yaml', 'configs/augmentation.yaml')

results = evaluate_cross_dataset(model, test_loader, device, head_type='edl', num_classes=4)
print("External Evaluation Metrics:")
for k, v in results.items():
    if k != 'confusion_matrix':
        print(f"  • {k}: {v:.4f}")

In [ ]:
# Cell 16 — Explainability
from src.explainability import compare_and_save_visualizations

# Attributions on a sample normal image using target layers from ResNet50
target_layers_gradcam = [model.backbone.layer4]
target_layers_layercam = [model.backbone.layer3]

# Find an image in Kermany folder
sample_image = 'datasets/Kermany/OCT2017 /test/NORMAL/NORMAL-1017326-1.jpeg'
if os.path.exists(sample_image):
    compare_and_save_visualizations(
        model=model,
        target_layers_gradcam=target_layers_gradcam,
        target_layers_layercam=target_layers_layercam,
        image_path=sample_image,
        target_class=3,
        output_dir='outputs/figures/explainability',
        prefix='normal_sample'
    )
else:
    print("Sample image path not found. Please provide a valid path.")

In [ ]:
# Cell 17 — Calibration
from src.evaluation import calculate_ece, calculate_brier_score
print("Calibration validation methods loaded and ready.")

In [ ]:
# Cell 18 — Generate All Paper Figures
import numpy as np
from src.plots import plot_confusion_matrix, plot_reliability_diagram

if 'results' in locals():
    # Generate confusion matrix
    plot_confusion_matrix(
        cm=results['confusion_matrix'],
        classes=["CNV", "DME", "DRUSEN", "NORMAL"],
        save_path='outputs/figures/confusion_matrix_EXP006.png'
    )
else:
    print("Run Cell 15 to calculate model outputs and generate figures.")

In [ ]:
# Cell 19 — Zip Results
!zip -r outputs.zip outputs/
!zip -r experiments.zip experiments/